# Flight Delays: preparazione e integrità
Prepariamo un unico contratto dati per i tre database, senza connessioni o ingestione. Il CSV originale contiene **31 colonne e nessuna singola chiave univoca**. Aggiungiamo `flight_id` da 1 a 5.819.079, seguendo i record nell'ordine originale.
La sorgente Kaggle `usdot/flight-delays`, versione 1, ha 592.406.591 byte (circa 564,96 MiB). Le righe identiche non vengono eliminate. I subset di 100k e 1M sono prefissi del completo, non nuovi campionamenti né sorgenti complete.
`Run All` mostra la configurazione senza preparare dati. Abilitare `ABILITA_PREPARAZIONE=True` solo per il setup del dataset, non durante la DEMO.


In [1]:
from pathlib import Path
import json
import logging
import sys
import yaml

# Jupyter può partire dalla root o dalla cartella notebooks.
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'config' / 'benchmark.yaml').is_file():
    raise RuntimeError('Aprire il notebook nel container del progetto o dalla root del repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.preprocessing import acquire_source, prepare_dataset
from src.validation import iter_prepared_chunks, validate_bundle
from src.schema import PREPARED_COLUMNS, SOURCE_SHA256

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
config = yaml.safe_load((ROOT / 'config' / 'benchmark.yaml').read_text(encoding='utf-8'))
settings = config['preprocessing']
RAW = ROOT / config['paths']['raw_csv']
OUTPUT = ROOT / config['paths']['prepared'] / settings['output_name']
SIZES = tuple(config['dataset_sizes'])
CHUNK_SIZE = settings['chunk_size']
OFFLINE = settings['offline']
FORCE = settings['force']
print({'source': str(RAW), 'output': str(OUTPUT), 'sizes': SIZES, 'chunk_size': CHUNK_SIZE, 'offline': OFFLINE, 'force': FORCE})
print('SHA-256 atteso della sorgente:', SOURCE_SHA256)
ABILITA_PREPARAZIONE = True  # La preparazione completa richiede tempo e spazio.


{'source': '/workspace/data/raw/flights.csv', 'output': '/workspace/data/prepared/flights-v1', 'sizes': (100000, 1000000, 5819079), 'chunk_size': 2000, 'offline': False, 'force': False}
SHA-256 atteso della sorgente: afa4e42188b113dd69427a3fbd00a4b6979c6235c866f963c2899d18a3403a83


## 1. Acquisizione esplicita e sorgente immutabile
Per default `offline: true`: se manca `data/raw/flights.csv`, questa cella si ferma con istruzioni, senza rete. Copiare prima il CSV completo, oppure impostare consapevolmente `offline: false` in YAML per scaricare il solo file dalla versione 1 tramite KaggleHub. La copia persistente elimina la dipendenza dalla cache.
Un file già presente non viene mai sovrascritto. SHA-256 e dimensioni sono registrati in `flights.source.json`; un checksum diverso dalla versione attesa causa un errore, non una correzione automatica.

In [2]:
if ABILITA_PREPARAZIONE is True:
    provenance = acquire_source(RAW, offline=OFFLINE)
    print(json.dumps(provenance, indent=2, ensure_ascii=False))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 191M/191M [00:11<00:00, 17.3MB/s] 

Extracting zip of flights.csv...


{
  "dataset": "usdot/flight-delays/versions/1",
  "url": "https://www.kaggle.com/datasets/usdot/flight-delays",
  "file": "flights.csv",
  "bytes": 592406591,
  "sha256": "afa4e42188b113dd69427a3fbd00a4b6979c6235c866f963c2899d18a3403a83",
  "acquisition_method": "kagglehub_versioned_download",
  "recorded_at_utc": "2026-11-08T14:50:41.939776+00:00",
  "contract": {
    "expected_rows": 5819079,
    "expected_sha256": "afa4e42188b113dd69427a3fbd00a4b6979c6235c866f963c2899d18a3403a83",
    "label": "kaggle-usdot-flight-delays-v1"
  }
}


## 2. Streaming, tipi e pubblicazione del bundle
`prepare_dataset` legge al massimo `CHUNK_SIZE` record originali per blocco, verifica i tipi e assegna il contatore fuori dal ciclo dei chunk. Conserva esattamente le 31 celle originali e aggiunge una sola colonna `flight_id`.
CSV UTF-8, intestazione `flight_id,YEAR,...,WEATHER_DELAY`, terminatori LF. Le celle vuote rappresentano `None` nel lettore tipizzato (`NULL` SQL / `null` Mongo). Zero e stringhe come `NA` restano valori distinti. Gli orari `HHMM` rimangono stringhe, inclusi gli zeri iniziali e `2400`; durate e ritardi sono interi nullable. I flag restano interi 0/1.
Ogni CSV viene riletto e confrontato cella per cella con il prefisso originale. La sequenza esatta 1..N prova unicità, assenza di ID nulli e ordine senza mantenere tutte le chiavi in RAM. Solo alla fine il bundle diventa visibile come directory validata. Anche preparando un solo subset si scansiona tutta la sorgente.

In [3]:
if ABILITA_PREPARAZIONE is True:
    result = prepare_dataset(RAW, OUTPUT, sizes=SIZES, chunk_size=CHUNK_SIZE, force=FORCE)
    manifest = result['manifest']
    print('Checkpoint riutilizzato:', result['skipped'])
    for item in manifest['artifacts']:
        print(item['file'], '| righe:', item['rows'], '| ID:', item['flight_id_min'], '..', item['flight_id_max'], '| SHA-256:', item['sha256'])
    print('Statistiche sorgente:', json.dumps(manifest['source']['statistics'], indent=2))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


2026-11-08 14:50:50,312 INFO Preprocessing: 100,000 record sorgente letti
2026-11-08 14:50:55,295 INFO Preprocessing: 200,000 record sorgente letti
2026-11-08 14:50:59,375 INFO Preprocessing: 300,000 record sorgente letti
2026-11-08 14:51:03,854 INFO Preprocessing: 400,000 record sorgente letti
2026-11-08 14:51:08,379 INFO Preprocessing: 500,000 record sorgente letti
2026-11-08 14:51:13,463 INFO Preprocessing: 600,000 record sorgente letti
2026-11-08 14:51:18,037 INFO Preprocessing: 700,000 record sorgente letti
2026-11-08 14:51:22,695 INFO Preprocessing: 800,000 record sorgente letti
2026-11-08 14:51:27,270 INFO Preprocessing: 900,000 record sorgente letti
2026-11-08 14:51:31,412 INFO Preprocessing: 1,000,000 record sorgente letti
2026-11-08 14:51:35,310 INFO Preprocessing: 1,100,000 record sorgente letti
2026-11-08 14:51:38,502 INFO Preprocessing: 1,200,000 record sorgente letti
2026-11-08 14:51:42,055 INFO Preprocessing: 1,300,000 record sorgente letti
2026-11-08 14:51:46,112 INFO P

Checkpoint riutilizzato: False
flights_100000.csv | righe: 100000 | ID: 1 .. 100000 | SHA-256: 59af1e9a0eb5d83ee79afc8ceda78e131a07cf4f6e27c5892b9092126ad172e6
flights_1000000.csv | righe: 1000000 | ID: 1 .. 1000000 | SHA-256: 24a239b6ad2a2169eff3fc32440c45922e7ed0d745b2095e296924b5edec27a2
flights_5819079.csv | righe: 5819079 | ID: 1 .. 5819079 | SHA-256: aed3ae98cf2610a68080d4c3d7e26067bbe55eff60e85eda83eb7f650db7a070
Statistiche sorgente: {
  "rows": 5819079,
  "null_counts": {
    "YEAR": 0,
    "MONTH": 0,
    "DAY": 0,
    "DAY_OF_WEEK": 0,
    "AIRLINE": 0,
    "FLIGHT_NUMBER": 0,
    "TAIL_NUMBER": 14721,
    "ORIGIN_AIRPORT": 0,
    "DESTINATION_AIRPORT": 0,
    "SCHEDULED_DEPARTURE": 0,
    "DEPARTURE_TIME": 86153,
    "DEPARTURE_DELAY": 86153,
    "TAXI_OUT": 89047,
    "WHEELS_OFF": 89047,
    "SCHEDULED_TIME": 6,
    "ELAPSED_TIME": 105071,
    "AIR_TIME": 105071,
    "DISTANCE": 0,
    "WHEELS_ON": 92513,
    "TAXI_IN": 92513,
    "SCHEDULED_ARRIVAL": 0,
    "ARRIVAL_TIME

## 3. Risorse osservate, non misure dei database
Il manifest registra il tempo di checksum, scrittura e validazione. L'acquisizione e il rename finale sono esclusi. La memoria è il massimo RSS del processo campionato ogni 100 ms con psutil, non la memoria di tutta Docker né un massimo matematico garantito. Nel notebook comprende anche il processo kernel.
Se compare `skipped=True`, le misure nel manifest sono quelle della preparazione precedente; `invocation_metrics` descrive invece il controllo checksum appena eseguito. Non mescolare questi due tempi.

In [4]:
if ABILITA_PREPARAZIONE is True:
    print('Preparazione salvata:', json.dumps(manifest['metrics'], indent=2))
    print('Chiamata corrente:', json.dumps(result['invocation_metrics'], indent=2))
    print('Picco RSS della preparazione (MiB):', round(manifest['metrics']['sampled_peak_rss_bytes'] / 2**20, 2))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


Preparazione salvata: {
  "elapsed_seconds": 461.36741175300006,
  "baseline_rss_bytes": 69201920,
  "sampled_peak_rss_bytes": 76390400,
  "rss_growth_bytes": 7188480,
  "sampling_interval_seconds": 0.1,
  "memory_method": "psutil RSS del processo, massimo campionato; non memoria totale Docker",
  "phase_seconds": {
    "source_checksum": 3.3553818800000954,
    "streaming_write": 225.74424373600004,
    "output_validation": 227.8599725649999
  },
  "scope": "checksum, scrittura e validazione; esclusi acquisizione e pubblicazione"
}
Chiamata corrente: {
  "elapsed_seconds": 461.36741175300006,
  "baseline_rss_bytes": 69201920,
  "sampled_peak_rss_bytes": 76390400,
  "rss_growth_bytes": 7188480,
  "sampling_interval_seconds": 0.1,
  "memory_method": "psutil RSS del processo, massimo campionato; non memoria totale Docker",
  "phase_seconds": {
    "source_checksum": 3.3553818800000954,
    "streaming_write": 225.74424373600004,
    "output_validation": 227.8599725649999
  },
  "scope": "

## 4. Verifica e lettura riutilizzabile
Lo skip verifica nuovamente SHA-256 della sorgente e di ogni output. Per ripetere tutti i controlli su celle/ID/tipi impostare `DEEP_RECHECK=True` sotto (richiede un'altra scansione dei file). L'esempio legge soltanto due record dal subset.
La mappatura SQL/BSON è salvata in `schema.json` e nel manifest: gli adapter la applicano esplicitamente, soprattutto `flight_id` -> BSON Int64 come `_id`. Qui non vengono creati schemi o documenti nei DB.

In [5]:
if ABILITA_PREPARAZIONE is True:
    DEEP_RECHECK = False
    checked = validate_bundle(OUTPUT, RAW, deep=DEEP_RECHECK, expected_sizes=tuple(sorted(SIZES)))
    print('Verifica completata:', checked['status'])
    sample_path = OUTPUT / manifest['artifacts'][0]['file']
    for row in next(iter_prepared_chunks(sample_path, chunk_size=2)):
        print(dict(zip(PREPARED_COLUMNS, row)))
else:
    print('Operazione disabilitata: ABILITA_PREPARAZIONE=False.')


Verifica completata: validated
{'flight_id': 1, 'YEAR': 2015, 'MONTH': 1, 'DAY': 1, 'DAY_OF_WEEK': 4, 'AIRLINE': 'AS', 'FLIGHT_NUMBER': 98, 'TAIL_NUMBER': 'N407AS', 'ORIGIN_AIRPORT': 'ANC', 'DESTINATION_AIRPORT': 'SEA', 'SCHEDULED_DEPARTURE': '0005', 'DEPARTURE_TIME': '2354', 'DEPARTURE_DELAY': -11, 'TAXI_OUT': 21, 'WHEELS_OFF': '0015', 'SCHEDULED_TIME': 205, 'ELAPSED_TIME': 194, 'AIR_TIME': 169, 'DISTANCE': 1448, 'WHEELS_ON': '0404', 'TAXI_IN': 4, 'SCHEDULED_ARRIVAL': '0430', 'ARRIVAL_TIME': '0408', 'ARRIVAL_DELAY': -22, 'DIVERTED': 0, 'CANCELLED': 0, 'CANCELLATION_REASON': None, 'AIR_SYSTEM_DELAY': None, 'SECURITY_DELAY': None, 'AIRLINE_DELAY': None, 'LATE_AIRCRAFT_DELAY': None, 'WEATHER_DELAY': None}
{'flight_id': 2, 'YEAR': 2015, 'MONTH': 1, 'DAY': 1, 'DAY_OF_WEEK': 4, 'AIRLINE': 'AA', 'FLIGHT_NUMBER': 2336, 'TAIL_NUMBER': 'N3KUAA', 'ORIGIN_AIRPORT': 'LAX', 'DESTINATION_AIRPORT': 'PBI', 'SCHEDULED_DEPARTURE': '0010', 'DEPARTURE_TIME': '0002', 'DEPARTURE_DELAY': -8, 'TAXI_OUT': 12, 

## Riutilizzo e gestione degli errori
Una seconda esecuzione con gli stessi subset riusa soltanto un checkpoint valido, senza riscrivere i CSV. Se un checksum differisce si ferma. `force: true` è una rigenerazione esplicita: conserva il vecchio bundle in una directory `.flights-v1.backup-...` e richiede spazio anche per la nuova copia. Non cancella l'originale.
Un processo interrotto non produce un checkpoint valido. Lock o staging residui vanno esaminati soltanto dopo aver verificato che non esista una preparazione ancora attiva. Nessuna ripresa automatica da un punto ambiguo del CSV.
La DEMO usa il dataset persistente: non eseguire preprocessing o download durante la dimostrazione. Il requisito dimensionale riguarda il sorgente completo, non il subset da 100k/1M. Qui non si eseguono ingestione o query.